# Research Layer Quickstart

Register labeled synthetic prompts, run strategy suites, and compare results either as a flat run table or grouped by prompt label. This stays entirely inside `packages/research/`.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "packages" / "research").exists() and (candidate / "packages" / "pipeline").exists():
        ROOT = candidate
        break
else:
    notebook_path = Path("packages/research/notebooks/quickstart.ipynb").resolve()
    for candidate in (notebook_path.parent, *notebook_path.parents):
        if (candidate / "packages" / "research").exists() and (candidate / "packages" / "pipeline").exists():
            ROOT = candidate
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from packages.research.experiment import (
    ExperimentRunner,
    TaskRegistry,
    TestPrompt,
    compare_runs,
    compare_runs_by_prompt_label,
    format_comparison_table,
    format_grouped_comparison_tables,
)
from packages.research.strategy.registry import list_strategies

print("Repo root:", ROOT)
print("Available strategies:", list_strategies())


Repo root: /Users/thorbthorb/Downloads/IL_ideation
Available strategies: ['grammar']


## Register a Labeled Prompt Suite

`TestPrompt` is intentionally tiny: `prompt` plus one label. The registry persists these prompts in the local SQLite research DB so the same suite can be reused across strategies and experiments.


In [2]:
runner = ExperimentRunner()
tasks = TaskRegistry(store=runner.store)

TEST_PROMPTS: list[TestPrompt] = [
    {
        "prompt": "quadruped robot that can climb stairs",
        "label": "morphlogy-provided",
    },
    {
        "prompt": "robot that should choose one best body plan for narrow warehouse aisles",
        "label": "infer-morphology-single",
    },
    {
        "prompt": "robot family for mixed terrain search and rescue",
        "label": "infer-morphology-family",
    },
    {
        "prompt": "invent a robot for moving fragile objects across clutter",
        "label": "open-ended",
    },
]

tasks.register_many(TEST_PROMPTS)
print("Registered prompts:")
for item in tasks.list():
    print(f"- [{item['label']}] {item['prompt']}")


Registered prompts:
- [morphlogy-provided] quadruped robot that can climb stairs
- [infer-morphology-single] robot that should choose one best body plan for narrow warehouse aisles
- [infer-morphology-family] robot family for mixed terrain search and rescue
- [open-ended] invent a robot for moving fragile objects across clutter
- [morphlogy-provided] quadruped robot with four articulated legs for climbing outdoor stairs while carrying a small sensor pack
- [morphlogy-provided] hexapod robot with six compliant legs for crossing loose gravel and shallow roots
- [morphlogy-provided] biped robot with two long legs for stepping over low warehouse obstacles
- [morphlogy-provided] humanoid robot with arms and legs for opening lightweight doors and walking through offices
- [morphlogy-provided] snake robot with a segmented body for inspecting narrow pipes with tight bends
- [morphlogy-provided] centipede robot with many short legs for crawling over rubble without tipping
- [morphlogy-provided]

## Run a Prompt Suite

`run_suite` runs every prompt against every strategy you list and carries the prompt label into each persisted run. Start with `grammar`; after the parametric strategy is registered later in this notebook, run it with the same `TEST_PROMPTS`.


In [3]:
EXPERIMENT_NAME = "notebook-demo"

runs = runner.run_suite(
    test_prompts=TEST_PROMPTS,
    strategy_names=["grammar"],
    experiment_name=EXPERIMENT_NAME,
    seed=42,
    max_candidates=3,
    extra={"max_attempts": 2},
)
result = runs[0]

print(f"Created {len(runs)} run(s)")
for run in runs:
    print(f"- {run.run_id} [{run.prompt_label}] strategy={run.strategy_name} error={run.error}")


Created 4 run(s)
- 60b7cd9f [morphlogy-provided] strategy=grammar error=None
- 4dba7719 [infer-morphology-single] strategy=grammar error=None
- e4264664 [infer-morphology-family] strategy=grammar error=None
- 34da5b90 [open-ended] strategy=grammar error=None


## Compare Runs: Flat and Grouped

The old comparison path still works: pass arbitrary runs to `compare_runs`. For prompt-suite analysis, use `compare_runs_by_prompt_label` and render one table per label.


In [4]:
store = runner.store
exp = store.find_experiment_by_name(EXPERIMENT_NAME)
if exp:
    all_runs = store.list_runs(exp.experiment_id)

    print("Flat comparison, label-aware:")
    print(format_comparison_table(compare_runs(all_runs, store), include_label=True))

    print("\nGrouped by prompt label:")
    grouped = compare_runs_by_prompt_label(all_runs, store)
    print(format_grouped_comparison_tables(grouped))
else:
    print("No experiment found - run the suite cell first.")


Flat comparison, label-aware:
Run ID      Strategy    Label       Designs     Compile%    Stable%     Score       Entropy     Time(s)   
----------  ----------  ----------  ----------  ----------  ----------  ----------  ----------  ----------
34da5b90    grammar     open-ended  3           100%        0%          0.000       0.00        268.6     
e4264664    grammar     infer-morphology-family  3           100%        0%          0.000       0.00        285.0     
4dba7719    grammar     infer-morphology-single  3           100%        0%          0.000       0.00        180.8     
60b7cd9f    grammar     morphlogy-provided  3           100%        0%          0.000       0.00        222.0     
a685d5be    parametric  open-ended  3           100%        0%          0.000       1.58        0.0       
375049eb    parametric  infer-morphology-family  3           100%        0%          0.000       1.58        0.0       
88dd0d5a    parametric  infer-morphology-single  3           100%  

## Inspect metrics

In [5]:
if result.metrics_report:
    m = result.metrics_report
    print(f"Compile rate: {m.compile_rate:.0%}")
    print(f"Mean screening score: {m.mean_screening_score:.3f}")
    print(f"Link count entropy: {m.link_count_entropy:.2f}")
    print(f"Wall time: {m.wall_time_seconds:.1f}s")
    print()
    for dm in m.per_design:
        print(f"  {dm.design_name}: links={dm.link_count} joints={dm.joint_count} compiles={dm.mjcf_compiles}")

Compile rate: 100%
Mean screening score: 0.000
Link count entropy: 0.00
Wall time: 222.0s

  grammar_42: links=20 joints=19 compiles=True
  grammar_43: links=20 joints=19 compiles=True
  grammar_44: links=20 joints=19 compiles=True


## Inspect design IR

In [6]:
if result.designs:
    ir = result.designs[0]
    print(f"Design: {ir.name}")
    print(f"Links: {[l.name for l in ir.links]}")
    print(f"Joints: {[j.name for j in ir.joints]}")
    print(f"Validation errors: {ir.validate()}")

Design: grammar_42
Links: ['S', 'BODY', 'CLIMBING_LEG_PAIR', 'TAIL', 'PELVIS', 'TORSO', 'pelvis_link', 'torso_link', 'SPINE_CHAIN', 'vertebra_link', 'CLIMBING_LEG', 'hip_joint', 'upper_leg', 'knee_joint', 'lower_leg', 'ankle_joint', 'adhesive_pad', 'tail_yaw', 'tail_link', 'tail_tip']
Joints: ['joint_S_to_BODY', 'joint_S_to_CLIMBING_LEG_PAIR', 'joint_S_to_TAIL', 'joint_BODY_to_PELVIS', 'joint_BODY_to_TORSO', 'joint_PELVIS_to_pelvis_link', 'joint_TORSO_to_torso_link', 'joint_TORSO_to_SPINE_CHAIN', 'joint_SPINE_CHAIN_to_vertebra_link', 'joint_CLIMBING_LEG_PAIR_to_CLIMBING_LEG', 'joint_CLIMBING_LEG_to_hip_joint', 'joint_CLIMBING_LEG_to_upper_leg', 'joint_CLIMBING_LEG_to_knee_joint', 'joint_CLIMBING_LEG_to_lower_leg', 'joint_CLIMBING_LEG_to_ankle_joint', 'joint_CLIMBING_LEG_to_adhesive_pad', 'joint_TAIL_to_tail_yaw', 'joint_TAIL_to_tail_link', 'joint_TAIL_to_tail_tip']
Validation errors: []


---
## Writing a Real Strategy: Parametric Morphology

The `GenerationStrategy` protocol is just `generate(prompt, config) -> list[RobotDesignIR]`.
Any algorithm fits — grammar search, template expansion, evolutionary, RL rollout.

This example builds robot IR **directly from parametric templates** — **no LLM calls**.
Each candidate varies structural parameters (leg segment count, body plan) so the benchmark
reports nonzero diversity entropy across the population.

**Design space explored per morphology family:**

| Family | Variation | Links | Joints |
|--------|-----------|-------|--------|
| quadruped | 1-seg legs (candidate 0) | 5 | 4 |
| quadruped | 2-seg legs (candidate 1) | 9 | 8 |
| quadruped | 3-seg legs (candidate 2) | 13 | 12 |
| hexapod | 6×2-seg legs | 13 | 12 |
| snake | N segments (4–9) | 4–9 | 3–8 |

A strategy owns its internal search loop. The experiment layer only calls `generate()`.

In [7]:
import random
from packages.pipeline.ir.design_ir import (
    RobotDesignIR, LinkIR, JointIR, JointType, JointLimits,
    Inertial, Visual, Geometry, Vector3, ActuatorSlot,
)
from packages.research.strategy.protocol import StrategyConfig
from packages.research.strategy.registry import register_strategy, list_strategies


# ── IR construction helpers ───────────────────────────────────────────────────

def _box_link(name, size, mass=1.0):
    g = Geometry(type='box', size=size)
    return LinkIR(
        name=name,
        inertial=Inertial(mass=mass, ixx=mass*0.01, iyy=mass*0.01, izz=mass*0.01),
        visual=Visual(geometry=g),
    )


def _capsule_link(name, radius, length, mass=0.5):
    g = Geometry(type='capsule', size=(radius, length))
    return LinkIR(
        name=name,
        inertial=Inertial(mass=mass, ixx=mass*0.005, iyy=mass*0.005, izz=mass*0.001),
        visual=Visual(geometry=g),
    )


def _rev_joint(name, parent, child, pos, axis, torque=30.0):
    """Revolute joint with position actuator."""
    return JointIR(
        name=name,
        joint_type=JointType.REVOLUTE,
        parent_link=parent,
        child_link=child,
        origin=pos,
        axis=axis,
        limits=JointLimits(lower=-1.047, upper=1.047, effort=torque, velocity=10.0),
        actuator=ActuatorSlot(actuator_type='position', max_torque=torque),
    )


# ── Family builders ───────────────────────────────────────────────────────────

def _build_quadruped(n_segments, idx):
    """
    Quadruped with configurable leg segment count.
      n_segments=1  ->  5 links,  4 joints  (hip only)
      n_segments=2  ->  9 links,  8 joints  (hip + thigh)
      n_segments=3  -> 13 links, 12 joints  (hip + thigh + shin)
    Varying n_segments across candidates produces nonzero link-count entropy.
    """
    s = f'_q{n_segments}s{idx}'
    links, joints = [], []

    torso = _box_link(f'torso{s}', size=(0.4, 0.2, 0.1), mass=5.0)
    links.append(torso)

    leg_attachments = [
        ('FL', Vector3( 0.15,  0.1, 0.0)),
        ('FR', Vector3( 0.15, -0.1, 0.0)),
        ('RL', Vector3(-0.15,  0.1, 0.0)),
        ('RR', Vector3(-0.15, -0.1, 0.0)),
    ]

    for leg_id, attach_pos in leg_attachments:
        hip = f'hip_{leg_id}{s}'
        links.append(_capsule_link(hip, radius=0.03, length=0.12, mass=0.4))
        joints.append(_rev_joint(
            f'hip_j_{leg_id}{s}', torso.name, hip,
            pos=attach_pos, axis=Vector3(0, 1, 0), torque=40.0,
        ))

        if n_segments >= 2:
            thigh = f'thigh_{leg_id}{s}'
            links.append(_capsule_link(thigh, radius=0.025, length=0.16, mass=0.3))
            joints.append(_rev_joint(
                f'knee_j_{leg_id}{s}', hip, thigh,
                pos=Vector3(0, 0, -0.06), axis=Vector3(0, 1, 0), torque=30.0,
            ))

        if n_segments >= 3:
            shin = f'shin_{leg_id}{s}'
            links.append(_capsule_link(shin, radius=0.02, length=0.14, mass=0.2))
            parent = thigh if n_segments >= 2 else hip
            joints.append(_rev_joint(
                f'ankle_j_{leg_id}{s}', parent, shin,
                pos=Vector3(0, 0, -0.08), axis=Vector3(0, 1, 0), torque=20.0,
            ))

    return RobotDesignIR(name=f'quadruped_{n_segments}seg_{idx}', links=links, joints=joints)


def _build_hexapod(idx):
    """Hexapod: 6 legs x 2 segments. Fixed structure — varies only by idx suffix."""
    s = f'_h{idx}'
    links, joints = [], []
    torso = _box_link(f'torso{s}', size=(0.5, 0.15, 0.08), mass=4.0)
    links.append(torso)

    leg_positions = [
        ('L1', Vector3( 0.18,  0.08, 0.0)),
        ('R1', Vector3( 0.18, -0.08, 0.0)),
        ('L2', Vector3( 0.00,  0.08, 0.0)),
        ('R2', Vector3( 0.00, -0.08, 0.0)),
        ('L3', Vector3(-0.18,  0.08, 0.0)),
        ('R3', Vector3(-0.18, -0.08, 0.0)),
    ]

    for leg_id, pos in leg_positions:
        coxa  = f'coxa_{leg_id}{s}'
        femur = f'femur_{leg_id}{s}'
        links.append(_capsule_link(coxa,  radius=0.02,  length=0.08,  mass=0.20))
        links.append(_capsule_link(femur, radius=0.018, length=0.10, mass=0.15))
        joints.append(_rev_joint(f'coxa_j_{leg_id}{s}',  torso.name, coxa,
                                  pos=pos,                  axis=Vector3(0, 1, 0), torque=25.0))
        joints.append(_rev_joint(f'femur_j_{leg_id}{s}', coxa, femur,
                                  pos=Vector3(0, 0, -0.04), axis=Vector3(0, 1, 0), torque=20.0))

    return RobotDesignIR(name=f'hexapod_{idx}', links=links, joints=joints)


def _build_snake(n_segments, idx):
    """
    Serial snake: alternating pitch/yaw joints along the spine.
    n_segments varies per candidate → varying link/joint counts → diversity.
    """
    s = f'_sn{n_segments}_{idx}'
    links, joints = [], []
    links.append(_capsule_link(f'seg_0{s}', radius=0.04, length=0.18, mass=0.5))
    for i in range(1, n_segments):
        seg = f'seg_{i}{s}'
        links.append(_capsule_link(seg, radius=0.04, length=0.18, mass=0.5))
        axis = Vector3(0, 1, 0) if i % 2 == 0 else Vector3(1, 0, 0)  # alternate pitch/yaw
        joints.append(_rev_joint(
            f'seg_j_{i}{s}', f'seg_{i-1}{s}', seg,
            pos=Vector3(0, 0, -0.09), axis=axis, torque=15.0,
        ))
    return RobotDesignIR(name=f'snake_{n_segments}seg_{idx}', links=links, joints=joints)


# ── Prompt classifier ─────────────────────────────────────────────────────────

_KEYWORDS = {
    'hexapod': ['hexapod', 'six-leg', 'insect', 'ant', 'spider'],
    'snake':   ['snake', 'worm', 'crawl', 'serpent', 'tube', 'eel'],
}

def _classify_prompt(prompt):
    p = prompt.lower()
    for family, keywords in _KEYWORDS.items():
        if any(k in p for k in keywords):
            return family
    return 'quadruped'


# ── Strategy ──────────────────────────────────────────────────────────────────

class ParametricMorphologyStrategy:
    """
    LLM-free strategy: deterministic IR construction from parametric templates.

    Algorithm (no API calls, reproducible at any seed):
      1. Classify prompt -> morphology family (quadruped / hexapod / snake)
      2. For each candidate index i, vary structural parameters:
           quadruped: n_segments = 1 + (i % 3)  -> 5/9/13 links per candidate
           snake:     n_segments = 4 + (i % 6)  -> 4-9 links per candidate
           hexapod:   fixed 13 links, idx suffix for name uniqueness
      3. Assemble LinkIR + JointIR directly, populate all Inertial/Visual/ActuatorSlot

    Compare this to GrammarStrategy, which runs an LLM agent loop to produce
    structural_rules then materializes IR via a second LLM call. Both satisfy
    the same GenerationStrategy protocol — the experiment layer doesn't care.
    """

    @property
    def name(self): return 'parametric'

    @property
    def version(self): return '1.0.0'

    def generate(self, prompt, config):
        family = _classify_prompt(prompt)
        return [self._build_candidate(family, i) for i in range(config.max_candidates)]

    def _build_candidate(self, family, idx):
        if family == 'hexapod':
            return _build_hexapod(idx)
        if family == 'snake':
            return _build_snake(n_segments=4 + (idx % 6), idx=idx)
        # quadruped: cycle through 1/2/3 segments for structural diversity
        return _build_quadruped(n_segments=1 + (idx % 3), idx=idx)


register_strategy('parametric', ParametricMorphologyStrategy)
print('Registered strategy:', ParametricMorphologyStrategy().name,
      'v' + ParametricMorphologyStrategy().version)
print('All strategies:', list_strategies())

Registered strategy: parametric v1.0.0
All strategies: ['grammar', 'parametric']


## Run the Parametric Strategy

No API key needed. Three structurally distinct quadruped candidates are built
in microseconds: 1-segment legs (5 links), 2-segment (9 links), 3-segment (13 links).

In [8]:
para_runs = runner.run_suite(
    test_prompts=TEST_PROMPTS,
    strategy_names=["parametric"],
    experiment_name=EXPERIMENT_NAME,
    seed=42,
    max_candidates=3,
)
para_result = para_runs[0]

print(f'Created {len(para_runs)} parametric run(s)')
print(f'First Run ID: {para_result.run_id}')
print(f'Designs: {len(para_result.designs)}')
print(f'Error:   {para_result.error}')


Created 4 parametric run(s)
First Run ID: 819d0380
Designs: 3
Error:   None


## Benchmark: Per-Design Breakdown

`evaluate_single` runs each IR through MJCF compilation and MuJoCo screening.
Key things to check:
- **IR valid**: joint parent/child links all resolve  
- **MJCF compiles**: passes MuJoCo load (requires MuJoCo installed)  
- **screening_score**: physics plausibility (stability, actuator coverage, no self-collision)  
- **link_count_entropy** > 0 confirms structural diversity across the population

In [9]:
from packages.research.benchmark.harness import evaluate_single

print('── Per-design ──────────────────────────────────────────')
for ir in para_result.designs:
    errs = ir.validate()
    m = evaluate_single(ir)
    status = 'COMPILES' if m.mjcf_compiles else 'FAILS   '
    print(f'{ir.name}')
    print(f'  links={m.link_count:<3} joints={m.joint_count:<3} actuated={m.actuated_joint_count:<3} [{status}]')
    print(f'  ir_valid={m.ir_valid}  screening_score={m.screening_score:.3f}')
    if errs:
        print(f'  VALIDATION ERRORS: {errs}')

if para_result.metrics_report:
    r = para_result.metrics_report
    print()
    print('── Aggregate ───────────────────────────────────────────')
    print(f'  compile_rate:        {r.compile_rate:.0%}')
    print(f'  mean_screening:      {r.mean_screening_score:.3f}')
    print(f'  link_count_entropy:  {r.link_count_entropy:.3f}  (> 0 = structural diversity)')
    print(f'  joint_count_entropy: {r.joint_count_entropy:.3f}')
    print(f'  morphology_families: {r.morphology_families}')
    print(f'  wall_time:           {r.wall_time_seconds:.3f}s  (no LLM = instant)')

── Per-design ──────────────────────────────────────────
quadruped_1seg_0
  links=5   joints=4   actuated=4   [COMPILES]
  ir_valid=True  screening_score=0.000
quadruped_2seg_1
  links=9   joints=8   actuated=8   [COMPILES]
  ir_valid=True  screening_score=0.000
quadruped_3seg_2
  links=13  joints=12  actuated=12  [COMPILES]
  ir_valid=True  screening_score=0.000

── Aggregate ───────────────────────────────────────────
  compile_rate:        100%
  mean_screening:      0.000
  link_count_entropy:  1.585  (> 0 = structural diversity)
  joint_count_entropy: 1.585
  morphology_families: {'quadruped': 3}
  wall_time:           0.000s  (no LLM = instant)


## Side-by-side Comparison: Parametric vs Grammar

Both runs landed in the same experiment (`notebook-demo`).  
`compare_runs` aggregates across all runs so you can see the tradeoffs:

| Dimension | Parametric | Grammar |
|-----------|------------|---------|
| Speed | Instant (no API) | Slow (LLM calls) |
| Cost | $0 | ~$0.02–0.10 per run |
| Diversity | Controlled by design | Emergent from LLM |
| Novelty | Template-bounded | Open-ended |

This is why the framework separates strategies from the experiment layer:
you can swap, combine, or ablate them without touching the benchmark code.

In [10]:
exp = runner.store.find_experiment_by_name(EXPERIMENT_NAME)
if exp:
    runs = runner.store.list_runs(exp.experiment_id)
    comparisons = compare_runs(runs, runner.store)
    print('Flat comparison across all runs:')
    print(format_comparison_table(comparisons, include_label=True))

    print('\nGrouped comparison by prompt label:')
    print(format_grouped_comparison_tables(compare_runs_by_prompt_label(runs, runner.store)))
else:
    print('No experiment found - run the cells above first.')


Flat comparison across all runs:
Run ID      Strategy    Label       Designs     Compile%    Stable%     Score       Entropy     Time(s)   
----------  ----------  ----------  ----------  ----------  ----------  ----------  ----------  ----------
ca16f6d9    parametric  open-ended  3           100%        0%          0.000       1.58        0.0       
ba500948    parametric  infer-morphology-family  3           100%        0%          0.000       1.58        0.0       
2f515ac8    parametric  infer-morphology-single  3           100%        0%          0.000       1.58        0.0       
819d0380    parametric  morphlogy-provided  3           100%        0%          0.000       1.58        0.0       
34da5b90    grammar     open-ended  3           100%        0%          0.000       0.00        268.6     
e4264664    grammar     infer-morphology-family  3           100%        0%          0.000       0.00        285.0     
4dba7719    grammar     infer-morphology-single  3           100

## What Makes a Good Strategy?

A strategy is **not** a prompt. It is an algorithm with a contract:

```python
def generate(self, prompt: str, config: StrategyConfig) -> list[RobotDesignIR]: ...
```

Good strategies to explore next:

| Strategy idea | Key insight |
|---------------|-------------|
| **EvolutionaryStrategy** | Mutate `RobotDesignIR` based on screening scores; no LLM needed |
| **GrammarStrategy** (built-in) | LLM produces structural rules; separate materializer converts to IR |
| **HybridStrategy** | Parametric seed population + LLM refinement of top-k candidates |
| **RetrievalStrategy** | Embed prompt, fetch closest design from SQLite store, perturb it |

All four satisfy the same protocol. Register with `register_strategy(name, cls)` and the
runner, benchmark harness, and comparison table work without modification.